In [1]:
%%time
# STEP 3 — Availability Gate
# Jupyter single-cell version

import polars as pl
from datetime import date
from pathlib import Path
from IPython.display import display

# ── Constants ────────────────────────────────────────────────────────────────

TODAY = date(2026, 6, 25)

FLOOR = 0.05

TARGET_CITIES = {
    "noida", "pune", "delhi", "new delhi", "delhi ncr",
    "gurgaon", "gurugram", "faridabad", "noida sector",
    "mumbai", "hyderabad",
}

RATE_FLOOR = 0.10
RATE_CEIL  = 0.90

# Engagement weights
W_RESPONSE  = 0.20
W_NOTICE    = 0.30
W_INTENT    = 0.25
W_INTERVIEW = 0.10
W_WORK_MODE = 0.15

OUTPUT_COLS = [
    "candidate_id",
    "availability_multiplier",
    "location_gate",
    "recency_gate",
    "engagement_score",
    "last_active_days_ago",
    "response_score",
    "notice_score",
    "intent_score",
    "interview_score",
    "work_mode_score",
    "response_rate_flag",
    "location_flag",
]

# ── Path resolver ────────────────────────────────────────────────────────────
# This makes the cell work whether your parquets are in:
#   artifacts/
#   dataset/artifacts/
#   dataset/

def find_file(*candidates: str) -> Path:
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(
        "Could not find any of these files:\n" +
        "\n".join(f"  - {c}" for c in candidates)
    )

SURVIVORS_PATH = find_file(
    "artifacts/survivors.parquet",
    "dataset/artifacts/survivors.parquet",
)

BASE_PATH = find_file(
    "artifacts/candidate_base.parquet",
    "dataset/artifacts/candidate_base.parquet",
    "dataset/candidate_base.parquet",
)

REDROB_PATH = find_file(
    "artifacts/candidate_redrob.parquet",
    "dataset/artifacts/candidate_redrob.parquet",
    "dataset/candidate_redrob.parquet",
)

# Write output next to survivors.parquet
OUTPUT_PATH = SURVIVORS_PATH.parent / "availability_scores.parquet"

print("Resolved paths:")
print(f"  survivors : {SURVIVORS_PATH}")
print(f"  base      : {BASE_PATH}")
print(f"  redrob    : {REDROB_PATH}")
print(f"  output    : {OUTPUT_PATH}")


# ── Load & Join ──────────────────────────────────────────────────────────────

def load_data() -> pl.DataFrame:
    survivors = pl.read_parquet(SURVIVORS_PATH).select("candidate_id")

    base = (
        pl.read_parquet(BASE_PATH)
        .select(["candidate_id", "location", "country"])
    )

    redrob = pl.read_parquet(REDROB_PATH)

    df = (
        survivors
        .join(base, on="candidate_id", how="left")
        .join(redrob, on="candidate_id", how="left")
    )

    return df


# ── Gate 1 — Location ────────────────────────────────────────────────────────

def compute_location_gate(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("location")
          .cast(pl.Utf8, strict=False)
          .str.to_lowercase()
          .str.strip_chars()
          .alias("_loc_norm"),

        pl.col("country")
          .cast(pl.Utf8, strict=False)
          .str.to_lowercase()
          .str.strip_chars()
          .alias("_country_norm"),
    )

    outside_india = (
        pl.col("_country_norm").is_not_null()
        & (pl.col("_country_norm") != "india")
    )

    both_null = (
        pl.col("_loc_norm").is_null()
        & pl.col("_country_norm").is_null()
    )

    in_target = pl.col("_loc_norm").is_in(list(TARGET_CITIES))

    relocate = pl.col("willing_to_relocate").fill_null(False)

    df = df.with_columns(
        pl.when(outside_india)
          .then(pl.lit(0.10))
          .when(both_null)
          .then(pl.lit(0.80))
          .when(in_target)
          .then(pl.lit(1.00))
          .when(relocate)
          .then(pl.lit(0.88))
          .otherwise(pl.lit(0.20))
          .cast(pl.Float64)
          .alias("location_gate")
    )

    return df.drop(["_loc_norm", "_country_norm"])


# ── Gate 2 — Recency ─────────────────────────────────────────────────────────

def compute_recency_gate(df: pl.DataFrame) -> pl.DataFrame:
    today_lit = pl.lit(TODAY)

    df = df.with_columns(
        pl.col("last_active_date")
          .cast(pl.Date, strict=False)
          .alias("_lad")
    )

    df = df.with_columns(
        pl.when(pl.col("_lad").is_null())
          .then(pl.lit(999, dtype=pl.Int64))
          .otherwise((today_lit - pl.col("_lad")).dt.total_days().cast(pl.Int64))
          .alias("last_active_days_ago")
    )

    days = pl.col("last_active_days_ago")

    df = df.with_columns(
        pl.when(days <= 14)
          .then(pl.lit(1.00))
          .when(days <= 60)
          .then(pl.lit(0.85))
          .when(days <= 180)
          .then(pl.lit(0.70))
          .when(days <= 365)
          .then(pl.lit(0.45))
          .otherwise(pl.lit(0.20))
          .cast(pl.Float64)
          .alias("recency_gate")
    )

    return df.drop("_lad")


# ── Engagement Signals ───────────────────────────────────────────────────────

def compute_response_score(df: pl.DataFrame) -> pl.DataFrame:
    clamped = pl.col("recruiter_response_rate").clip(RATE_FLOOR, RATE_CEIL)

    return df.with_columns(
        pl.when(pl.col("recruiter_response_rate").is_null())
          .then(pl.lit(0.90))
          .when(clamped < 0.20)
          .then(pl.lit(0.30))
          .when(clamped < 0.40)
          .then(pl.lit(0.65))
          .when(clamped <= 0.60)
          .then(pl.lit(0.85))
          .otherwise(pl.lit(1.00))
          .cast(pl.Float64)
          .alias("response_score")
    )


def compute_notice_score(df: pl.DataFrame) -> pl.DataFrame:
    days = pl.col("notice_period_days")

    return df.with_columns(
        pl.when(days.is_null())
          .then(pl.lit(0.80))
          .when(days <= 30)
          .then(pl.lit(1.00))
          .when(days <= 60)
          .then(pl.lit(0.95))
          .when(days <= 90)
          .then(pl.lit(0.85))
          .otherwise(pl.lit(0.75))
          .cast(pl.Float64)
          .alias("notice_score")
    )


def compute_intent_score(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("open_to_work_flag")
          .cast(pl.Utf8, strict=False)
          .str.to_lowercase()
          .str.strip_chars()
          .alias("_otw_str")
    )

    df = df.with_columns(
        pl.when(pl.col("_otw_str").is_in(["true", "1"]))
          .then(pl.lit(1.00))
          .otherwise(pl.lit(0.60))
          .cast(pl.Float64)
          .alias("intent_score")
    )

    return df.drop("_otw_str")


def compute_interview_score(df: pl.DataFrame) -> pl.DataFrame:
    clamped = pl.col("interview_completion_rate").clip(RATE_FLOOR, RATE_CEIL)

    return df.with_columns(
        pl.when(pl.col("interview_completion_rate").is_null())
          .then(pl.lit(1.00))
          .when(clamped >= 0.80)
          .then(pl.lit(1.00))
          .when(clamped >= 0.50)
          .then(pl.lit(0.90))
          .otherwise(pl.lit(0.75))
          .cast(pl.Float64)
          .alias("interview_score")
    )


def compute_work_mode_score(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("preferred_work_mode")
          .cast(pl.Utf8, strict=False)
          .str.to_lowercase()
          .str.strip_chars()
          .alias("_wm_norm")
    )

    df = df.with_columns(
        pl.when(pl.col("_wm_norm").is_null())
          .then(pl.lit(0.90))
          .when(pl.col("_wm_norm").is_in(["onsite", "hybrid", "flexible"]))
          .then(pl.lit(1.00))
          .when(pl.col("_wm_norm") == "remote")
          .then(pl.lit(0.65))
          .otherwise(pl.lit(0.90))
          .cast(pl.Float64)
          .alias("work_mode_score")
    )

    return df.drop("_wm_norm")


def compute_engagement_score(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        (
            pl.col("response_score")  * W_RESPONSE  +
            pl.col("notice_score")    * W_NOTICE    +
            pl.col("intent_score")    * W_INTENT    +
            pl.col("interview_score") * W_INTERVIEW +
            pl.col("work_mode_score") * W_WORK_MODE
        ).alias("engagement_score")
    )


# ── Combined Multiplier + Flags ──────────────────────────────────────────────

def compute_multiplier(df: pl.DataFrame) -> pl.DataFrame:
    raw = (
        pl.col("location_gate") *
        pl.col("recency_gate") *
        pl.col("engagement_score")
    )

    return df.with_columns(
        pl.max_horizontal([raw, pl.lit(FLOOR)])
          .alias("availability_multiplier")
    )


def compute_flags(df: pl.DataFrame) -> pl.DataFrame:
    clamped_rr = pl.col("recruiter_response_rate").clip(RATE_FLOOR, RATE_CEIL)

    return df.with_columns(
        pl.when(pl.col("recruiter_response_rate").is_null())
          .then(pl.lit(False))
          .when(clamped_rr < 0.40)
          .then(pl.lit(True))
          .otherwise(pl.lit(False))
          .alias("response_rate_flag"),

        (pl.col("location_gate") < 0.50)
          .alias("location_flag"),
    )


# ── Summary ──────────────────────────────────────────────────────────────────

def print_summary(df: pl.DataFrame) -> None:
    total = len(df)

    outside_india = df.filter(pl.col("location_gate") == 0.10).height
    loc_mismatch  = df.filter(pl.col("location_gate") == 0.20).height
    inactive      = df.filter(pl.col("last_active_days_ago") > 180).height
    low_response  = df.filter(pl.col("response_rate_flag")).height

    ghost_count = df.filter(
        (pl.col("last_active_days_ago") > 180)
        | pl.col("response_rate_flag")
        | (pl.col("location_gate") == 0.20)
    ).height

    ghost_rate = ghost_count / total if total else 0

    am = pl.col("availability_multiplier")
    strong     = df.filter(am >= 0.80).height
    moderate   = df.filter((am >= 0.50) & (am < 0.80)).height
    weak       = df.filter((am >= 0.20) & (am < 0.50)).height
    near_floor = df.filter((am > FLOOR) & (am < 0.20)).height
    at_floor   = df.filter(am == FLOOR).height
    floor_pct  = at_floor / total * 100 if total else 0

    print()
    print("=" * 70)
    print("  STEP 3 — AVAILABILITY GATE SUMMARY")
    print("=" * 70)
    print(f"  Total processed          : {total:>10,}")
    print(f"  Outside India            : {outside_india:>10,}")
    print(f"  Location mismatch        : {loc_mismatch:>10,}  (not in target, won't relocate)")
    print(f"  Inactive (> 180 days)    : {inactive:>10,}")
    print(f"  Low response rate        : {low_response:>10,}  (< 0.40 after clamping)")
    print(f"  Ghost rate               : {ghost_rate:>10.1%}  (inactive | low_response | loc_mismatch)")
    print()
    print("  Multiplier distribution:")
    print(f"    1.00 – 0.80  (strong)     : {strong:>8,}  ({strong / total:.1%})")
    print(f"    0.80 – 0.50  (moderate)   : {moderate:>8,}  ({moderate / total:.1%})")
    print(f"    0.50 – 0.20  (weak)       : {weak:>8,}  ({weak / total:.1%})")
    print(f"    0.20 – 0.05  (near-floor) : {near_floor:>8,}  ({near_floor / total:.1%})")
    print(f"    0.05         (floor)      : {at_floor:>8,}  ({floor_pct:.1f}%)")
    print()

    if ghost_rate < 0.05:
        print("  ⚠ WARNING: ghost rate < 5% — signals may be too lenient")
    if ghost_rate > 0.60:
        print("  ⚠ WARNING: ghost rate > 60% — signals may be too aggressive")
    if floor_pct > 10.0:
        print(f"  ⚠ WARNING: {floor_pct:.1f}% of candidates hit the floor — Step 3 acting as hard drop")

    print("=" * 70)
    print()


# ── Run Step 3 ────────────────────────────────────────────────────────────────

print(f"\nLoading data... TODAY = {TODAY}")
df = load_data()
print(f"Rows after survivor + base + redrob join: {len(df):,}")

print("\nComputing location + recency gates...")
df = compute_location_gate(df)
df = compute_recency_gate(df)

print("Computing engagement scores...")
df = compute_response_score(df)
df = compute_notice_score(df)
df = compute_intent_score(df)
df = compute_interview_score(df)
df = compute_work_mode_score(df)
df = compute_engagement_score(df)

print("Computing final multiplier + flags...")
df = compute_multiplier(df)
df = compute_flags(df)

out = df.select(OUTPUT_COLS)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.write_parquet(OUTPUT_PATH)

print(f"\nWrote output → {OUTPUT_PATH}")

print_summary(out)

# ── Display useful notebook outputs ──────────────────────────────────────────

print("Output schema:")
display(out.schema)

print("\nFirst 10 rows:")
display(out.head(10).to_pandas())

print("\nTop 10 strongest availability candidates:")
display(
    out.sort("availability_multiplier", descending=True)
       .head(10)
       .to_pandas()
)

print("\nBottom 10 weakest availability candidates:")
display(
    out.sort("availability_multiplier")
       .head(10)
       .to_pandas()
)

print("\nWorked-example check: CAND_0000001")
cand_check = out.filter(pl.col("candidate_id") == "CAND_0000001")
display(cand_check.to_pandas())

if cand_check.height > 0:
    val = cand_check["availability_multiplier"][0]
    print(f"CAND_0000001 availability_multiplier = {val:.6f}")
    print("Expected from handover ≈ 0.077 if the source data matches.")
else:
    print("CAND_0000001 not found in survivors/output.")

Resolved paths:
  survivors : dataset\artifacts\survivors.parquet
  base      : dataset\artifacts\candidate_base.parquet
  redrob    : dataset\artifacts\candidate_redrob.parquet
  output    : dataset\artifacts\availability_scores.parquet

Loading data... TODAY = 2026-06-25
Rows after survivor + base + redrob join: 99,975

Computing location + recency gates...
Computing engagement scores...
Computing final multiplier + flags...

Wrote output → dataset\artifacts\availability_scores.parquet

  STEP 3 — AVAILABILITY GATE SUMMARY
  Total processed          :     99,975
  Outside India            :     24,881
  Location mismatch        :     53,426  (not in target, won't relocate)
  Inactive (> 180 days)    :     29,400
  Low response rate        :     44,202  (< 0.40 after clamping)
  Ghost rate               :      81.3%  (inactive | low_response | loc_mismatch)

  Multiplier distribution:
    1.00 – 0.80  (strong)     :        0  (0.0%)
    0.80 – 0.50  (moderate)   :   10,019  (10.0%)
  

<timed exec>:164: DeprecationWarning: Casting from String to Date is deprecated and will be removed in Polars 2.0.
Use `str.to_date()` instead.


Schema([('candidate_id', String),
        ('availability_multiplier', Float64),
        ('location_gate', Float64),
        ('recency_gate', Float64),
        ('engagement_score', Float64),
        ('last_active_days_ago', Int64),
        ('response_score', Float64),
        ('notice_score', Float64),
        ('intent_score', Float64),
        ('interview_score', Float64),
        ('work_mode_score', Float64),
        ('response_rate_flag', Boolean),
        ('location_flag', Boolean)])


First 10 rows:


,candidate_id,availability_multiplier,location_gate,recency_gate,engagement_score,last_active_days_ago,response_score,notice_score,intent_score,interview_score,work_mode_score,response_rate_flag,location_flag
0,CAND_0000001,0.076925,0.10,0.85,0.9050,36,0.65,0.95,1.0,0.90,1.00,True,True
1,CAND_0000002,0.081450,0.20,0.45,0.9050,225,0.65,0.95,1.0,0.90,1.00,True,True
2,CAND_0000003,0.055650,0.10,0.70,0.7950,96,0.85,0.75,0.6,1.00,1.00,False,True
3,CAND_0000004,0.051100,0.10,0.70,0.7300,92,0.65,0.75,0.6,0.75,1.00,True,True
4,CAND_0000005,0.364320,0.88,0.45,0.9200,267,0.65,1.00,1.0,0.90,1.00,True,False
5,CAND_0000006,0.050000,0.10,0.70,0.6225,117,0.30,0.75,0.6,0.90,0.65,True,True
6,CAND_0000007,0.654500,0.88,0.85,0.8750,31,1.00,1.00,0.6,0.75,1.00,False,False
7,CAND_0000008,0.073350,0.20,0.45,0.8150,194,0.85,0.85,0.6,0.90,1.00,False,True
8,CAND_0000009,0.051275,0.10,0.70,0.7325,149,0.85,0.75,0.6,0.90,0.65,False,True
9,CAND_0000010,0.066725,0.10,0.85,0.7850,57,0.85,0.75,0.6,0.90,1.00,False,True


...


Bottom 10 weakest availability candidates:


,candidate_id,availability_multiplier,location_gate,recency_gate,engagement_score,last_active_days_ago,response_score,notice_score,intent_score,interview_score,work_mode_score,response_rate_flag,location_flag
0,CAND_0000006,0.05,0.1,0.70,0.6225,117,0.30,0.75,0.6,0.90,0.65,True,True
1,CAND_0000028,0.05,0.1,0.70,0.6925,86,0.30,0.95,0.6,1.00,0.65,True,True
2,CAND_0000037,0.05,0.1,0.45,0.9000,196,1.00,1.00,0.6,1.00,1.00,False,True
3,CAND_0000042,0.05,0.1,0.45,0.8450,245,0.85,1.00,0.6,0.75,1.00,False,True
4,CAND_0000056,0.05,0.1,0.45,0.8300,207,0.85,0.95,0.6,0.75,1.00,False,True
5,CAND_0000064,0.05,0.1,0.45,0.8050,217,0.65,0.95,0.6,0.90,1.00,True,True
6,CAND_0000071,0.05,0.1,0.70,0.6600,162,0.30,0.75,0.6,0.75,1.00,True,True
7,CAND_0000089,0.05,0.1,0.45,0.8000,264,1.00,0.75,0.6,0.75,1.00,False,True
8,CAND_0000094,0.05,0.1,0.45,0.7775,182,0.85,0.95,0.6,0.75,0.65,False,True
9,CAND_0000117,0.05,0.1,0.45,0.8150,208,0.85,0.85,0.6,0.90,1.00,False,True



Worked-example check: CAND_0000001


,candidate_id,availability_multiplier,location_gate,recency_gate,engagement_score,last_active_days_ago,response_score,notice_score,intent_score,interview_score,work_mode_score,response_rate_flag,location_flag
0,CAND_0000001,0.076925,0.1,0.85,0.905,36,0.65,0.95,1.0,0.9,1.0,True,True


CAND_0000001 availability_multiplier = 0.076925
Expected from handover ≈ 0.077 if the source data matches.
CPU times: total: 1.94 s
Wall time: 3.09 s
